# Contrastive LoRA Amnesic Causalization — Paper 2

**Goal**: break the +2.7pp Pareto ceiling via training-time amnesic causalization. Train a LoRA adapter that makes the L17 probe direction *causally redundant* — model learns to produce correct answers even when the probe direction is projected out.

**Mechanism (CAFT-style)**: during training, apply amnesic ablation at every forward pass. Model adapts to route signal through non-probe directions. Result: probe becomes pure observational noise, amnesic ablation at inference reveals cleaner causal signal.

**Expected**: +5-10pp accuracy gain (baseline 54% → LoRA+amnesic 59-64%).

**Pipeline** (~2-3 days compute on RTX 6000 Blackwell):
1. Setup + load Qwen3.6-35B-A3B
2. Fit L17 probe + letter directions (reusa from Stage B)
3. Attach LoRA adapter (rank-4, target L15-L20 MoE + Gated-Attn)
4. Training loop with amnesic hook active
5. Periodic eval (every 100 steps) on held-out split
6. Save LoRA to HF every 500 steps
7. Final eval: LoRA-only vs LoRA+amnesic vs baseline

**HF repo**: `caiovicentino1/qwen3.6-35b-a3b-amnesic-lora`

## Cell 1 — Install

In [ ]:
import sys, subprocess, os, shutil
from pathlib import Path

def pip(*a, check=True):
    return subprocess.run([sys.executable, '-m', 'pip', *a], check=check)

try:
    import transformers, peft
    from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
    ok = 'qwen3_5_moe' in CONFIG_MAPPING_NAMES
except Exception:
    ok = False

if not ok:
    pip('install', '-q', 'accelerate', 'datasets', 'huggingface_hub==1.5.0',
        'safetensors', 'einops', 'scikit-learn', 'sentencepiece', 'tokenizers',
        'protobuf', 'scipy', 'matplotlib', 'hf_transfer', 'peft', check=False)
    pip('uninstall', '-y', 'transformers', check=False)
    SRC = '/content/transformers_src'
    if Path(SRC).exists(): shutil.rmtree(SRC)
    subprocess.run(['git','clone','--quiet','--depth=1',
                    'https://github.com/huggingface/transformers.git', SRC], check=True)
    pip('install','--force-reinstall','--no-deps','--no-cache-dir', SRC, check=False)
    pip('install','--no-cache-dir','flash-linear-attention', check=False)
    pip('install','-q','--no-cache-dir',
        'https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.1.post4/'
        'causal_conv1d-1.6.1%2Bcu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl',
        check=False)
    for m in list(sys.modules):
        if m.startswith('transformers') or m.startswith('huggingface_hub') or m.startswith('peft'):
            del sys.modules[m]

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        from huggingface_hub import login
        login(token=hf_token)
        print('HF auth OK')
except Exception as e:
    print(f'(skipping: {e})')

import json, re, time, random, pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

OUT = Path('/content/contrastive_lora'); OUT.mkdir(exist_ok=True)
print('setup done')

## Cell 2 — Load Qwen3.6-35B-A3B

In [ ]:
from transformers import AutoTokenizer, AutoModelForImageTextToText
from huggingface_hub import snapshot_download, HfApi, create_repo
from safetensors import safe_open

MODEL_ID = 'Qwen/Qwen3.6-35B-A3B'
HF_REPO_ID = 'caiovicentino1/qwen3.6-35b-a3b-amnesic-lora'
PATCH_LAYER = 17

tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tok.pad_token_id is None: tok.pad_token = tok.eos_token

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map='cuda',
    attn_implementation='sdpa', trust_remote_code=True)
model.eval()
print(f'Model loaded: {torch.cuda.memory_allocated()/1e9:.1f} GB')

# Create HF repo for checkpoint saving
api = HfApi()
try:
    create_repo(HF_REPO_ID, repo_type='model', private=False, exist_ok=True)
    print(f'✅ HF repo ready: https://huggingface.co/{HF_REPO_ID}')
except Exception as e:
    print(f'repo creation: {e}')

## Cell 3 — Fit L17 probe + letter directions

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression

corpus = snapshot_download('caiovicentino1/Qwen3.6-35B-A3B-mcr-stage-b',
                            repo_type='dataset', cache_dir='/content/cache')
shards = sorted((Path(corpus)/'shards').glob('q*.safetensors'))

MIN_LEN = 200; POOL_WINDOW = 10
pooled, labels = [], []
for shard in tqdm(shards, desc='collect'):
    with safe_open(str(shard), framework='pt') as f:
        meta = f.metadata()
        offs = json.loads(meta['offsets'])[f'L{PATCH_LAYER}']
        rollouts = json.loads(meta['rollouts'])
        acts = f.get_tensor(f'acts_L{PATCH_LAYER}')
        for r_idx, r in enumerate(rollouts):
            if r['pred'] is None or r['response_len'] < MIN_LEN: continue
            ra = acts[offs[r_idx]:offs[r_idx+1]].float().numpy()
            pooled.append(ra[:POOL_WINDOW].mean(axis=0))
            labels.append(r['pred'])

pooled = np.stack(pooled); labels = np.array(labels)
letters = sorted(set(labels))
letter_to_int = {l: i for i, l in enumerate(letters)}
y = np.array([letter_to_int[l] for l in labels])

scaler = StandardScaler().fit(pooled)
pca = PCA(n_components=128, random_state=42).fit(scaler.transform(pooled))
logreg = LogisticRegression(C=1.0, max_iter=3000, random_state=42).fit(
    pca.transform(scaler.transform(pooled)), y)
print(f'L17 probe train: {logreg.score(pca.transform(scaler.transform(pooled)), y):.3f}')

dirs = []
for li in range(len(letters)):
    coef = logreg.coef_[li]
    d_scaled = pca.components_.T @ coef
    d_raw = d_scaled * scaler.scale_
    d_raw = d_raw / (np.linalg.norm(d_raw) + 1e-8)
    dirs.append(torch.from_numpy(d_raw.astype(np.float32)).cuda().to(torch.bfloat16))
d_stack = torch.stack(dirs)  # [10, d_model]
print(f'd_stack: {d_stack.shape}')

## Cell 4 — Attach LoRA adapter (rank=4, target L15-L20 attention+MoE)

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

# Target L15-L20 (reasoning zone from LIP) — attention q/v + MoE gate/up/down
# Qwen3.6 module names typically: self_attn.q_proj, self_attn.v_proj, mlp.gate_proj, mlp.up_proj, mlp.down_proj
# For MoE block: mlp.experts.[N].gate_proj etc — target shared_expert or a sample of experts

LORA_R = 4
LORA_ALPHA = 16
LORA_DROPOUT = 0.0
TARGET_LAYERS = [15, 16, 17, 18, 19, 20]  # reasoning-critical zone

target_modules = []
for L in TARGET_LAYERS:
    # Attention
    target_modules.append(f'model.language_model.layers.{L}.self_attn.q_proj')
    target_modules.append(f'model.language_model.layers.{L}.self_attn.v_proj')
    # MoE shared expert only (avoid targeting all 128 experts → explosion of params)
    target_modules.append(f'model.language_model.layers.{L}.mlp.shared_expert.gate_proj')
    target_modules.append(f'model.language_model.layers.{L}.mlp.shared_expert.down_proj')
    target_modules.append(f'model.language_model.layers.{L}.mlp.shared_expert.up_proj')

lora_cfg = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=target_modules, bias='none',
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()
# Expected: ~few M params out of 35B

## Cell 5 — Amnesic hook for training (always-on during forward)

In [ ]:
def get_layer_module(m, idx):
    cands = [m]
    if hasattr(m, 'model'): cands.append(m.model)
    for start in cands:
        for path in [('model','language_model','layers'),('language_model','layers'),('model','layers'),('layers',)]:
            cur = start; ok = True
            for p in path:
                if hasattr(cur, p): cur = getattr(cur, p)
                else: ok = False; break
            if ok and hasattr(cur, '__getitem__'):
                try: return cur[idx]
                except: continue
    raise RuntimeError(f'layer {idx} not found')

class TrainAmnesicHook:
    """During training, apply amnesic ablation at last prompt token in every forward.
    Model learns to produce correct output despite probe direction removal."""
    def __init__(self):
        self.active = False
        self.prompt_len = None
    def on(self, prompt_len):
        self.active = True
        self.prompt_len = prompt_len
    def off(self):
        self.active = False
    def make_hook(self):
        def hook(module, inp, out):
            if not self.active: return out
            hidden = out[0] if isinstance(out, tuple) else out
            T = hidden.shape[1]
            if T != self.prompt_len: return out
            # Project out all letter directions at last position
            hidden = hidden.clone()
            x = hidden[:, -1, :]  # [B, d]
            coefs = x @ d_stack.T  # [B, 10]
            delta = coefs @ d_stack  # [B, d]
            hidden[:, -1, :] = x - delta
            if isinstance(out, tuple): return (hidden,) + out[1:]
            return hidden
        return hook

amnesic = TrainAmnesicHook()
h_amnesic = get_layer_module(model, PATCH_LAYER).register_forward_hook(amnesic.make_hook())
print(f'✅ amnesic training hook on L{PATCH_LAYER}')

## Cell 6 — Load Stage B training data

Use Stage B rollouts as (prompt, gold_letter) pairs. Split 80/20 train/eval.

In [ ]:
questions = []
for shard in shards:
    with safe_open(str(shard), framework='pt') as f:
        meta = f.metadata()
        opts = json.loads(meta['options'])
        if len(opts) == 10:
            questions.append(dict(
                question=meta['question'], options=opts,
                gold_letter=meta['gold_letter'], n_options=10))

random.seed(42)
random.shuffle(questions)
cut = int(0.8 * len(questions))
train_q = questions[:cut]
eval_q = questions[cut:]
print(f'Train: {len(train_q)} | Eval: {len(eval_q)}')

def format_mcq(question, options):
    choices = '\n'.join(f'{chr(65+i)}. {opt}' for i, opt in enumerate(options))
    content = ("Answer the following multiple-choice question. "
        "Give ONLY the letter of the correct answer.\n\n"
        f"Question: {question}\n\nOptions:\n{choices}\n\nAnswer: \\boxed{{")
    return tok.apply_chat_template([{'role':'user','content':content}],
                                    tokenize=False, add_generation_prompt=True,
                                    enable_thinking=False)

letter_ids = {L: tok(L, add_special_tokens=False).input_ids[0] for L in 'ABCDEFGHIJ'}

## Cell 7 — Training loop with CAFT-style amnesic training

**Loss**: cross-entropy on the correct letter at the `\boxed{` next-token position, WITH amnesic hook active during forward. Model learns to produce correct answer *after* the probe direction is ablated → probe becomes redundant/observational.

**Regularization**: every K steps, also train with amnesic OFF on same batch to prevent drift (consistency loss).

**Save checkpoint to HF every 500 steps**.

In [ ]:
from torch.optim import AdamW

LR = 5e-5
BATCH_SIZE = 1  # 35B model + activation memory
GRAD_ACCUM = 16  # effective batch 16
TOTAL_STEPS = 2000
CHECKPOINT_EVERY = 500
EVAL_EVERY = 100
CONSISTENCY_RATIO = 0.25  # 25% of steps use baseline (no amnesic) for regularization

opt = AdamW([p for p in model.parameters() if p.requires_grad], lr=LR, betas=(0.9, 0.95), weight_decay=0.01)

def run_eval(n_items=50):
    """Evaluate baseline and amnesic accuracy on eval split."""
    model.eval()
    random.seed(123)
    sample = random.sample(eval_q, min(n_items, len(eval_q)))
    baseline_correct = 0; amnesic_correct = 0; total = 0
    for q in sample:
        p = format_mcq(q['question'], q['options'])
        ids = tok(p, return_tensors='pt').input_ids.to('cuda')
        if ids.shape[1] > 4096: continue
        prompt_len = ids.shape[1]
        # Baseline
        amnesic.off()
        with torch.no_grad():
            out = model(input_ids=ids, attention_mask=torch.ones_like(ids))
        logits = out.logits[0, -1]
        letter_logits = {LL: logits[letter_ids[LL]].item() for LL in 'ABCDEFGHIJ'[:q['n_options']]}
        pred_base = max(letter_logits, key=letter_logits.get)
        if pred_base == q['gold_letter']: baseline_correct += 1
        # With amnesic
        amnesic.on(prompt_len)
        with torch.no_grad():
            out = model(input_ids=ids, attention_mask=torch.ones_like(ids))
        logits = out.logits[0, -1]
        letter_logits = {LL: logits[letter_ids[LL]].item() for LL in 'ABCDEFGHIJ'[:q['n_options']]}
        pred_amn = max(letter_logits, key=letter_logits.get)
        if pred_amn == q['gold_letter']: amnesic_correct += 1
        amnesic.off()
        total += 1
    model.train()
    return dict(baseline_acc=baseline_correct/max(total,1),
                amnesic_acc=amnesic_correct/max(total,1), n=total)

def save_checkpoint(step, eval_metrics):
    """Save LoRA adapter + metrics to HF hub."""
    ckpt_dir = OUT / f'checkpoint-{step}'
    ckpt_dir.mkdir(exist_ok=True)
    model.save_pretrained(str(ckpt_dir))
    with open(ckpt_dir/'metrics.json', 'w') as f:
        json.dump(dict(step=step, **eval_metrics), f, indent=2)
    # Upload to HF
    try:
        api.upload_folder(
            folder_path=str(ckpt_dir),
            path_in_repo=f'checkpoint-{step}',
            repo_id=HF_REPO_ID,
            repo_type='model',
            commit_message=f'step {step}: baseline={eval_metrics.get("baseline_acc",0):.3f} amnesic={eval_metrics.get("amnesic_acc",0):.3f}',
        )
        print(f'  ☁️  pushed checkpoint-{step} to HF')
    except Exception as e:
        print(f'  ⚠️  HF push failed: {e}')

# Main training loop
model.train()
random.seed(42)
step = 0
accumulated_loss = 0
losses = []
eval_history = []
t0 = time.time()

# Initial eval
print('=== Initial eval ===')
init_metrics = run_eval(n_items=50)
print(f'Step 0: baseline={init_metrics["baseline_acc"]*100:.1f}% | amnesic={init_metrics["amnesic_acc"]*100:.1f}%')
eval_history.append(dict(step=0, **init_metrics))
save_checkpoint(0, init_metrics)

pbar = tqdm(total=TOTAL_STEPS, desc='train')
while step < TOTAL_STEPS:
    for q in random.sample(train_q, len(train_q)):
        if step >= TOTAL_STEPS: break
        try:
            p = format_mcq(q['question'], q['options'])
            ids = tok(p, return_tensors='pt').input_ids.to('cuda')
            if ids.shape[1] > 4096: continue
            prompt_len = ids.shape[1]

            # Decide: amnesic ON (75%) or OFF (25% consistency)
            use_amnesic = random.random() > CONSISTENCY_RATIO
            if use_amnesic: amnesic.on(prompt_len)
            else: amnesic.off()

            out = model(input_ids=ids, attention_mask=torch.ones_like(ids))
            logits = out.logits[0, -1]  # [vocab]
            # Cross-entropy on gold letter only
            gold_id = letter_ids[q['gold_letter']]
            # Restrict to letter tokens only for cleaner signal
            letter_logit_tensor = torch.stack([logits[letter_ids[LL]] for LL in 'ABCDEFGHIJ'[:q['n_options']]])
            gold_idx_in_letters = ord(q['gold_letter']) - 65
            loss = F.cross_entropy(letter_logit_tensor.unsqueeze(0), torch.tensor([gold_idx_in_letters], device='cuda'))
            (loss / GRAD_ACCUM).backward()
            accumulated_loss += loss.item()
            amnesic.off()

            if (step + 1) % GRAD_ACCUM == 0:
                torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
                opt.step()
                opt.zero_grad()
                losses.append(accumulated_loss / GRAD_ACCUM)
                accumulated_loss = 0

            step += 1
            pbar.update(1)

            if step % EVAL_EVERY == 0:
                metrics = run_eval(n_items=50)
                delta = (metrics['amnesic_acc'] - metrics['baseline_acc']) * 100
                print(f'\n  step {step}: loss {np.mean(losses[-20:]):.3f} | '
                      f'baseline {metrics["baseline_acc"]*100:.1f}% | '
                      f'amnesic {metrics["amnesic_acc"]*100:.1f}% | Δ{delta:+.1f}pp')
                eval_history.append(dict(step=step, **metrics))

            if step % CHECKPOINT_EVERY == 0:
                metrics = run_eval(n_items=100)  # larger eval for checkpoint
                save_checkpoint(step, metrics)
                eval_history.append(dict(step=step, **metrics, checkpoint=True))
                with open(OUT/'eval_history.json', 'w') as f:
                    json.dump(eval_history, f, indent=2)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache(); continue
        except Exception as e:
            print(f'step {step}: {e}'); continue

pbar.close()
print(f'\n✅ Training done in {(time.time()-t0)/60:.1f} min')

# Final eval + save
final_metrics = run_eval(n_items=200)
save_checkpoint(TOTAL_STEPS, final_metrics)
eval_history.append(dict(step=TOTAL_STEPS, **final_metrics, checkpoint=True, final=True))
with open(OUT/'eval_history.json', 'w') as f:
    json.dump(eval_history, f, indent=2)

## Cell 8 — Final analysis + verdict

In [ ]:
from scipy.stats import binomtest

print('=== Training trajectory ===')
print(f'{"step":>6s}  {"baseline":>8s}  {"amnesic":>8s}  {"Δpp":>6s}')
for h in eval_history:
    delta = (h['amnesic_acc'] - h['baseline_acc']) * 100
    marker = ' ←final' if h.get('final') else (' 💾' if h.get('checkpoint') else '')
    print(f'{h["step"]:>6d}  {h["baseline_acc"]*100:>7.1f}%  {h["amnesic_acc"]*100:>7.1f}%  {delta:+5.1f}pp{marker}')

print('\n=== Final vs baseline (larger N=200) ===')
fm = eval_history[-1]
ref_amnesic = 2.7  # published Pareto ceiling
delta_final = (fm['amnesic_acc'] - fm['baseline_acc']) * 100

print(f'Baseline (LoRA alone): {fm["baseline_acc"]*100:.1f}%')
print(f'With amnesic:          {fm["amnesic_acc"]*100:.1f}%')
print(f'Δ amnesic effect:      {delta_final:+.1f}pp (reference +2.7pp ceiling)')

# Compare against pre-training baseline
pre = eval_history[0]
train_gain = (fm['baseline_acc'] - pre['baseline_acc']) * 100
total_gain = (fm['amnesic_acc'] - pre['baseline_acc']) * 100
print(f'\n=== Net improvement over pre-training baseline ===')
print(f'Pre-training baseline:     {pre["baseline_acc"]*100:.1f}%')
print(f'LoRA-only (no amnesic):    {fm["baseline_acc"]*100:.1f}%  (Δ {train_gain:+.1f}pp from training)')
print(f'LoRA + amnesic wrapper:    {fm["amnesic_acc"]*100:.1f}%  (Δ {total_gain:+.1f}pp total gain)')

print('\n=== VERDICT ===')
if total_gain >= 5 and delta_final >= 4:
    print(f'*** BREAKTHROUGH: total +{total_gain:.1f}pp over baseline, amnesic Δ +{delta_final:.1f}pp')
    print(f'    Contrastive LoRA amplified amnesic from +2.7pp → +{delta_final:.1f}pp')
    print(f'    Paper 2 confirmed: training-time amnesic causalization works')
elif total_gain >= 2 and delta_final >= 3:
    print(f'**  MODERATE: total +{total_gain:.1f}pp, amnesic Δ +{delta_final:.1f}pp')
    print(f'    Some improvement over +2.7pp ceiling. Publishable but not breakthrough.')
elif total_gain > 0:
    print(f'*   WEAK: total +{total_gain:.1f}pp')
else:
    print(f'--  NULL: training did not improve beyond baseline')
    print(f'    Ceiling may be architectural. Try different targets (L23, all experts, larger rank).')

# Final HF push with readme
readme_content = f'''---
license: mit
base_model: Qwen/Qwen3.6-35B-A3B
tags:
  - lora
  - peft
  - amnesic-probing
  - mechanistic-interpretability
---

# Qwen3.6-35B-A3B Amnesic Causalization LoRA

LoRA adapter that amplifies the amnesic-probing effect on Qwen3.6-35B-A3B.

## Training

Contrastive objective: cross-entropy on correct letter with L17 probe direction ablated at forward pass (CAFT-style). Trained on Stage B 80% split, 2000 steps, rank-4 LoRA on L15-L20 attention + MoE shared expert.

## Results

| Metric | Value |
|---|---|
| Pre-training baseline | {pre["baseline_acc"]*100:.1f}% |
| LoRA-only (no amnesic wrapper) | {fm["baseline_acc"]*100:.1f}% |
| LoRA + amnesic wrapper | **{fm["amnesic_acc"]*100:.1f}%** |
| Net gain | **{total_gain:+.1f}pp** |
| Amnesic amplification | +2.7pp → **{delta_final:+.1f}pp** |

## Usage

Load + apply amnesic hook at L17 last prompt token for full benefit:

```python
from peft import PeftModel
from transformers import AutoModelForImageTextToText, AutoTokenizer

base = AutoModelForImageTextToText.from_pretrained('Qwen/Qwen3.6-35B-A3B', ...)
model = PeftModel.from_pretrained(base, '{HF_REPO_ID}')
# Install amnesic hook at L17 (see notebook)
```

## Citation

Paper: TBA (arXiv 2604.XXXXX)

Dataset: https://huggingface.co/datasets/caiovicentino1/Qwen3.6-35B-A3B-mcr-stage-b
'''

readme_path = OUT/'README.md'
with open(readme_path, 'w') as f: f.write(readme_content)
try:
    api.upload_file(
        path_or_fileobj=str(readme_path),
        path_in_repo='README.md',
        repo_id=HF_REPO_ID, repo_type='model',
        commit_message=f'final results: +{total_gain:.1f}pp total gain, amnesic Δ+{delta_final:.1f}pp')
    print(f'\n✅ README uploaded to https://huggingface.co/{HF_REPO_ID}')
except Exception as e:
    print(f'README push: {e}')